# Face Recognition Model — Trained From Scratch
### For the Employee Attendance Scanner (school project)

This notebook trains a small **Convolutional Neural Network from scratch** (no pretrained weights, no transfer learning) that recognizes **your face** so you don't need to scan your QR code a second time.

**How it fits into the attendance system:**

```
Google Colab (this notebook)              GitHub Pages site                Google Apps Script
┌──────────────────────────┐    export    ┌───────────────────────┐  POST  ┌──────────────────┐
│ 1. Capture your face via │ ───────────▶ │ model/ folder          │ ─────▶ │ action: faceScan │
│    the Colab webcam      │  TensorFlow  │ (model.json + weights) │        │ writes Time Out  │
│ 2. Train a CNN (3 class) │     .js      │ face.js runs the model │        │ to the Google    │
│ 3. Convert to TF.js      │              │ in the browser, live   │        │ Spreadsheet      │
└──────────────────────────┘              └───────────────────────┘        └──────────────────┘
```

**The 3 classes** (folder names matter — they must stay in this alphabetical order, it is the same order used by `js/face.js`):

| # | Class | What it contains | Where it comes from |
|---|-------------|--------------------------------------|------------------------------|
| 0 | `my_face` | Photos of YOU | Captured below with webcam |
| 1 | `no_face` | Empty room / background / no person | Captured below with webcam |
| 2 | `other_face` | Faces of other people | LFW public research dataset |

**Tips for a good model:** run the `my_face` capture cell 2–3 times in different lighting, with/without glasses, different angles and distances. More variety = better real-world accuracy.

> ⚠️ Use a normal Colab runtime (CPU is fine, GPU is faster). Allow webcam access when the browser asks.

In [ ]:
# @title 1. Setup: imports and dataset folders
import os, glob, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print('TensorFlow version:', tf.__version__)

DATASET_DIR = 'dataset'
CLASSES = ['my_face', 'no_face', 'other_face']  # alphabetical — do not reorder
IMG_SIZE = 96

for c in CLASSES:
    os.makedirs(os.path.join(DATASET_DIR, c), exist_ok=True)

def count_images():
    for c in CLASSES:
        n = len(glob.glob(f'{DATASET_DIR}/{c}/*.jpg'))
        print(f'{c:12s}: {n} images')

count_images()

In [ ]:
# @title 2. Webcam burst-capture helper (Colab -> Python bridge)
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def capture_burst(label, count=100, interval_ms=300, quality=0.85):
    """Shows your webcam, captures `count` frames, saves them as JPGs
    into dataset/<label>/. Run multiple times to append more images."""
    js = Javascript('''
      async function captureBurst(count, intervalMs, quality) {
        const div = document.createElement('div');
        const video = document.createElement('video');
        const status = document.createElement('h3');
        status.textContent = 'Starting camera...';
        video.style.width = '480px';
        div.appendChild(status);
        div.appendChild(video);
        document.body.appendChild(div);

        const stream = await navigator.mediaDevices.getUserMedia({ video: true });
        video.srcObject = stream;
        await video.play();
        await new Promise(r => setTimeout(r, 800)); // let exposure settle

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        const frames = [];
        for (let i = 0; i < count; i++) {
          canvas.getContext('2d').drawImage(video, 0, 0);
          frames.push(canvas.toDataURL('image/jpeg', quality));
          status.textContent = 'Capturing ' + (i + 1) + ' / ' + count + ' — move your head slowly!';
          await new Promise(r => setTimeout(r, intervalMs));
        }
        stream.getTracks().forEach(t => t.stop());
        div.remove();
        return frames;
      }
    ''')
    display(js)
    frames = eval_js(f'captureBurst({count}, {interval_ms}, {quality})')

    out_dir = os.path.join(DATASET_DIR, label)
    start = len(os.listdir(out_dir))
    for i, data_url in enumerate(frames):
        with open(f'{out_dir}/img_{start + i:04d}.jpg', 'wb') as f:
            f.write(b64decode(data_url.split(',')[1]))
    print(f'Saved {len(frames)} images to {out_dir}/')
    count_images()

## 3. Collect YOUR face (`my_face`)
Sit in front of the webcam. While it captures, **slowly turn your head left/right, tilt up/down, move closer/farther, smile / neutral face**.

Run this cell **2–3 times** (it appends): once in normal light, once with different lighting, once with/without glasses if you wear them. Aim for **200–300 total**.

In [ ]:
capture_burst('my_face', count=100, interval_ms=300)

## 4. Collect background (`no_face`)
Now **step out of the frame** (or point the camera at the wall/desk/ceiling and move it around). The model needs to learn what "nobody here" looks like. Aim for **~150 images**.

In [ ]:
capture_burst('no_face', count=150, interval_ms=200)

In [ ]:
# @title 5. Collect OTHER people's faces (`other_face`) from the LFW research dataset
# LFW = "Labeled Faces in the Wild", a classic public face dataset.
# First download is ~200 MB and takes a few minutes.
from sklearn.datasets import fetch_lfw_people
from PIL import Image

lfw = fetch_lfw_people(color=True, resize=1.0)
images = lfw.images
if images.max() <= 1.0:
    images = images * 255.0
print('LFW images available:', images.shape)

N_OTHER = 600
rng = np.random.RandomState(42)
picks = rng.choice(len(images), size=min(N_OTHER, len(images)), replace=False)

out_dir = os.path.join(DATASET_DIR, 'other_face')
for i, idx in enumerate(picks):
    arr = np.clip(images[idx], 0, 255).astype('uint8')
    img = Image.fromarray(arr)
    # center-crop to a square, then resize — matches how the browser crops frames
    w, h = img.size
    side = min(w, h)
    img = img.crop(((w - side) // 2, (h - side) // 2, (w + side) // 2, (h + side) // 2))
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img.save(f'{out_dir}/lfw_{i:04d}.jpg', quality=90)

print(f'Saved {len(picks)} other_face images.')
count_images()

In [ ]:
# @title 6. Crop your face out of the webcam frames (Haar cascade)
# The raw webcam frames contain a lot of background. We crop a square around
# your detected face (with margin) so the model learns YOUR FACE, not your room.
# Frames where no face is found are deleted (you were probably out of frame).
import cv2

cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

kept, dropped = 0, 0
for path in sorted(glob.glob(f'{DATASET_DIR}/my_face/*.jpg')):
    img = cv2.imread(path)
    if img is None:
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))
    if len(faces) == 0:
        os.remove(path)
        dropped += 1
        continue
    # largest detected face
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    # expand the box ~35% so it looks like the browser's oval-guide crop
    cx, cy = x + w / 2, y + h / 2
    side = int(max(w, h) * 1.35)
    x0 = max(0, int(cx - side / 2)); y0 = max(0, int(cy - side / 2))
    x1 = min(img.shape[1], x0 + side); y1 = min(img.shape[0], y0 + side)
    crop = cv2.resize(img[y0:y1, x0:x1], (IMG_SIZE, IMG_SIZE))
    cv2.imwrite(path, crop)
    kept += 1

print(f'Cropped {kept} face images, removed {dropped} frames without a detectable face.')
count_images()

In [ ]:
# @title 7. Build the training / validation datasets (with augmentation)
BATCH_SIZE = 32
SEED = 1337

train_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='categorical')
val_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, label_mode='categorical')

print('Class order:', train_ds.class_names)
assert train_ds.class_names == CLASSES, 'Class order must match js/face.js CLASS_NAMES!'

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
])

AUTOTUNE = tf.data.AUTOTUNE
# augment (training only), then scale pixels to 0..1 — js/face.js does the same /255
train_ds = (train_ds
            .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .map(lambda x, y: (x / 255.0, y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds = (val_ds
          .map(lambda x, y: (x / 255.0, y), num_parallel_calls=AUTOTUNE)
          .prefetch(AUTOTUNE))

# preview a training batch
plt.figure(figsize=(10, 4))
for imgs, labels in train_ds.take(1):
    for i in range(min(8, len(imgs))):
        plt.subplot(2, 4, i + 1)
        plt.imshow(imgs[i].numpy())
        plt.title(CLASSES[int(np.argmax(labels[i]))], fontsize=9)
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# @title 8. Define the CNN — built from scratch, no pretrained weights
def build_model(input_size=IMG_SIZE, num_classes=len(CLASSES)):
    inputs = keras.Input(shape=(input_size, input_size, 3))
    x = layers.Conv2D(16, 3, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)                      # 48x48
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)                      # 24x24
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)                      # 12x12
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='face_cnn')

model = build_model()
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# @title 9. Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ])

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.title('Accuracy'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Loss'); plt.legend()
plt.show()

In [ ]:
# @title 10. Evaluate: confusion matrix on the validation set
from sklearn.metrics import confusion_matrix, classification_report

y_true, y_pred = [], []
for imgs, labels in val_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=CLASSES))
cm = confusion_matrix(y_true, y_pred)
plt.imshow(cm, cmap='Blues')
plt.xticks(range(len(CLASSES)), CLASSES, rotation=20)
plt.yticks(range(len(CLASSES)), CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion matrix')
plt.colorbar(); plt.show()

# What matters most for this project: my_face precision.
# If 'my_face' gets confused with 'other_face', capture more images and retrain.

In [ ]:
# @title 11. Live sanity test with one webcam photo
def take_test_photo():
    js = Javascript('''
      async function takeTestPhoto() {
        const div = document.createElement('div');
        const video = document.createElement('video');
        const btn = document.createElement('button');
        btn.textContent = 'Capture test photo';
        video.style.width = '480px';
        div.appendChild(video); div.appendChild(btn);
        document.body.appendChild(div);
        const stream = await navigator.mediaDevices.getUserMedia({ video: true });
        video.srcObject = stream; await video.play();
        await new Promise(r => btn.onclick = r);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth; canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getTracks().forEach(t => t.stop());
        div.remove();
        return canvas.toDataURL('image/jpeg', 0.9);
      }
    ''')
    display(js)
    data_url = eval_js('takeTestPhoto()')
    with open('test_photo.jpg', 'wb') as f:
        f.write(b64decode(data_url.split(',')[1]))

take_test_photo()

from PIL import Image
img = Image.open('test_photo.jpg')
w, h = img.size
side = int(min(w, h) * 0.72)  # same center-crop ratio js/face.js uses
img = img.crop(((w - side) // 2, (h - side) // 2, (w + side) // 2, (h + side) // 2))
img = img.resize((IMG_SIZE, IMG_SIZE))

x = np.asarray(img, dtype='float32')[None, ...] / 255.0
probs = model.predict(x, verbose=0)[0]
plt.imshow(img); plt.axis('off')
plt.title(' | '.join(f'{c}: {p:.2f}' for c, p in zip(CLASSES, probs)))
plt.show()

In [ ]:
# @title 12. Export to TensorFlow.js and download
model.save('face_model.h5')

!pip install -q tensorflowjs
!tensorflowjs_converter --input_format=keras face_model.h5 tfjs_model

# --- Patch Keras-3 quirks so tf.loadLayersModel accepts the file ---
# Keras 3 writes 'batch_shape', DTypePolicy dtype objects, and a new
# inbound_nodes format; the TF.js browser runtime expects the Keras 2 forms.
import json
with open('tfjs_model/model.json', encoding='utf-8') as f:
    mj = json.load(f)
mcfg = mj['modelTopology']['model_config']['config']
for layer in mcfg['layers']:
    lc = layer.get('config', {})
    if 'batch_shape' in lc:
        lc['batch_input_shape'] = lc.pop('batch_shape')
    if isinstance(lc.get('dtype'), dict):
        lc['dtype'] = lc['dtype'].get('config', {}).get('name', 'float32')
    nodes = layer.get('inbound_nodes', [])
    if nodes and isinstance(nodes[0], dict):
        layer['inbound_nodes'] = [
            [[a['config']['keras_history'][0], a['config']['keras_history'][1],
              a['config']['keras_history'][2], {}]
             for a in n.get('args', [])
             if isinstance(a, dict) and a.get('class_name') == '__keras_tensor__']
            for n in nodes]
for key in ('input_layers', 'output_layers'):
    v = mcfg.get(key)
    if isinstance(v, list) and v and isinstance(v[0], str):
        mcfg[key] = [v]
with open('tfjs_model/model.json', 'w', encoding='utf-8') as f:
    json.dump(mj, f)
print('model.json patched for TF.js browser compatibility')

!ls -lh tfjs_model
!zip -r -q face_model_tfjs.zip tfjs_model

from google.colab import files
files.download('face_model_tfjs.zip')

# If the converter errors on your Colab runtime, use the Python API instead:
#   import tensorflowjs as tfjs
#   tfjs.converters.save_keras_model(model, 'tfjs_model')
# and if pip changed the TensorFlow version: Runtime -> Restart session,
# then re-run only cell 1 imports and this cell's convert lines (the model
# is already saved in face_model.h5).

## 13. Connect the model to your attendance website

1. **Unzip** `face_model_tfjs.zip`. Inside `tfjs_model/` you'll find `model.json` and one or more `group1-shard*.bin` weight files.
2. **Copy those files** (the files, not the folder) into the `model/` folder of the Employee-Attendance-Scanner repo, so the site sees `model/model.json`.
3. In `js/config.js`, set `FACE.EMPLOYEE_ID` to the employee ID you registered for yourself (e.g. `'EMP001'`).
4. Commit + push. GitHub Pages now serves the model as static files.
5. Open **scanner.html → Face Mode (AI)**:
   - `face.js` downloads `model/model.json` with TensorFlow.js and runs it **in the browser** — no server, no Colab needed at runtime.
   - Every ~400 ms it center-crops the webcam frame (the oval guide), resizes it to 96×96, divides by 255 (same preprocessing as cell 7!) and predicts.
   - After 4 consecutive frames of `my_face` ≥ 90% confidence, it POSTs `{action: 'faceScan', employee_id, confidence}` to the Apps Script web app, which writes Break Out / Break In / Time Out into the Google Spreadsheet.
   - **Time In always requires the QR scan** — the backend rejects face-only time-ins, so the face feature exactly covers "don't scan the QR a second time".

**Tuning:** if the scanner is too strict, lower `FACE.MIN_CONFIDENCE` (e.g. 0.8) in `js/config.js`; if it's too lax, raise it (0.95) or capture more training photos and retrain.

**Honest limitation (worth mentioning in your report):** a printed photo of your face can fool a simple CNN like this — real systems add liveness detection. That, plus requiring QR for the first scan, is a good discussion point for the project defense.